
# Notebook 21 — Residual Fixed Points and Finite-Size Universality Collapse

Notebook 20 showed that residual topology trajectories form shared universality structure.

Notebook 21 asks:

```text
Do residual universality trajectories converge toward fixed points as graph size increases?
```

Core frame:

```text
Residual topology classes can be studied as finite-size flows toward topology-specific or branch-specific fixed-point estimates.
```

This notebook is Colab-safe and self-contained:
- it loads Notebook 20 trajectory outputs if available,
- otherwise it rebuilds compatible residual trajectory data internally,
- it estimates topology-level fixed points,
- it estimates branch-level fixed points,
- it exports figures, CSVs, JSON, markdown notes, and an optional zip/download block.

Language note:
- this notebook uses “fixed-point estimate,” “approaching,” and “finite-size trend”
- it does not claim a mathematical proof of fixed points.


In [ ]:

import json
import zipfile
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

np.random.seed(42)

# ---------------------------------------------------
# robust repo detection for Colab and local Jupyter
# ---------------------------------------------------

def detect_repo_root():
    cwd = Path.cwd()

    if (cwd / "notebooks").exists() or (cwd / ".git").exists():
        return cwd

    if cwd.name == "notebooks":
        return cwd.parent

    search_root = Path("/content") if Path("/content").exists() else cwd

    for name in [
        "residual_phase_trajectory_embedding.csv",
        "universality_similarity_matrix.csv",
        "universality_summary.csv",
        "residual_geometry_features.csv",
        "residual_classification_feature_matrix.csv",
    ]:
        matches = list(search_root.rglob(name))
        if matches:
            if matches[0].parent.name == "results":
                return matches[0].parent.parent
            return matches[0].parents[1]

    return Path("/content") if Path("/content").exists() else cwd

REPO_ROOT = detect_repo_root()
RESULTS_DIR = REPO_ROOT / "results"
FIG_DIR = REPO_ROOT / "figures"
DOCS_DIR = REPO_ROOT / "docs"

RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)

GRAPH_SIZES = [16, 32, 64, 128]

TOPOLOGIES = [
    "ring_lattice",
    "small_world",
    "erdos_renyi",
    "scale_free",
    "modular_clustered",
]

TOPOLOGY_LABELS = {
    "ring_lattice": "ring lattice",
    "small_world": "small world",
    "erdos_renyi": "Erdős–Rényi",
    "scale_free": "scale free",
    "modular_clustered": "modular clustered",
}

print("cwd:", Path.cwd())
print("repo root:", REPO_ROOT)
print("results dir:", RESULTS_DIR)
print("figures dir:", FIG_DIR)
print("docs dir:", DOCS_DIR)


## 1. Load trajectory data or regenerate compatible inputs

In [ ]:

# ---------------------------------------------------
# fallback synthetic residual-geometry model
# consistent with Notebooks 19–20 self-contained paths
# ---------------------------------------------------

NOISE_GRID = np.linspace(0.0, 0.40, 81)
MIDPOINT_LEVEL = 0.50

def logistic_z(z):
    return 1 / (1 + np.exp(-z))

def shared_profile(z):
    return 1 - logistic_z(z)

TOPOLOGY_PARAMS = {
    "ring_lattice": {
        "eta_inf": 0.135, "eta_shift": 0.060,
        "sigma_inf": 0.030, "sigma_scale": 0.080,
        "nu": 0.45, "beta": 0.40,
        "fragment_strength": 0.08, "modifier": 0.98,
    },
    "small_world": {
        "eta_inf": 0.160, "eta_shift": 0.075,
        "sigma_inf": 0.038, "sigma_scale": 0.095,
        "nu": 0.48, "beta": 0.37,
        "fragment_strength": 0.06, "modifier": 1.02,
    },
    "erdos_renyi": {
        "eta_inf": 0.120, "eta_shift": 0.055,
        "sigma_inf": 0.028, "sigma_scale": 0.075,
        "nu": 0.42, "beta": 0.45,
        "fragment_strength": 0.10, "modifier": 0.96,
    },
    "scale_free": {
        "eta_inf": 0.105, "eta_shift": 0.052,
        "sigma_inf": 0.025, "sigma_scale": 0.070,
        "nu": 0.44, "beta": 0.50,
        "fragment_strength": 0.14, "modifier": 0.93,
    },
    "modular_clustered": {
        "eta_inf": 0.092, "eta_shift": 0.048,
        "sigma_inf": 0.023, "sigma_scale": 0.065,
        "nu": 0.40, "beta": 0.52,
        "fragment_strength": 0.18, "modifier": 0.90,
    },
}

def finite_size_eta_c(params, N):
    return params["eta_inf"] + params["eta_shift"] * (N ** (-params["nu"]))

def finite_size_sigma(params, N):
    return params["sigma_inf"] + params["sigma_scale"] * (N ** (-params["beta"]))

def simulate_cgcs_curve(topology, N, noise_grid, repeat=0):
    p = TOPOLOGY_PARAMS[topology]
    eta_c = finite_size_eta_c(p, N)
    sigma = finite_size_sigma(p, N)

    z = (noise_grid - eta_c) / sigma
    base = shared_profile(z)

    central_weight = np.exp(-0.5 * z**2)
    outside_weight = 1 - central_weight
    fragment = (
        p["fragment_strength"]
        * outside_weight
        * logistic_z((noise_grid - eta_c) / (2.0 * sigma))
    )

    rng = np.random.default_rng(
        40_000 + repeat + N + sum(ord(c) for c in topology)
    )
    noise_term = rng.normal(0, 0.010 * np.sqrt(32 / N), size=len(noise_grid))

    return np.clip(p["modifier"] * base - fragment + noise_term, 0, 1)

def extract_transition_metrics(noise, cgcs):
    noise = np.asarray(noise, dtype=float)
    cgcs = np.asarray(cgcs, dtype=float)

    midpoint_idx = int(np.argmin(np.abs(cgcs - MIDPOINT_LEVEL)))
    eta_mid = float(noise[midpoint_idx])

    deriv = np.gradient(cgcs, noise)
    max_abs_slope = float(np.max(np.abs(deriv)))
    sigma_est = float(1 / max(4 * max_abs_slope, 1e-6))

    return eta_mid, sigma_est

def normalized_entropy_from_energy(z, energy, bins=24):
    hist, _ = np.histogram(z, bins=bins, range=(-6, 6), weights=energy)
    total = hist.sum()
    if total <= 0:
        return 0.0
    p = hist / total
    p = p[p > 0]
    return float(-np.sum(p * np.log(p)) / np.log(bins))

def regenerate_geometry_features():
    rows = []
    repeats = 24

    for N in GRAPH_SIZES:
        for topology in TOPOLOGIES:
            curves = []
            for repeat in range(repeats):
                curves.append(simulate_cgcs_curve(topology, N, NOISE_GRID, repeat))

            mean_curve = np.array(curves).mean(axis=0)
            eta_mid, sigma_est = extract_transition_metrics(NOISE_GRID, mean_curve)
            sigma_est = max(sigma_est, 1e-6)

            z = (NOISE_GRID - eta_mid) / sigma_est
            predicted = shared_profile(z)
            residual = mean_curve - predicted

            energy = residual ** 2
            abs_res = np.abs(residual)

            left_energy = float(np.sum(energy[z < 0]))
            right_energy = float(np.sum(energy[z >= 0]))
            lr_total = left_energy + right_energy

            k = max(1, int(np.ceil(0.10 * len(energy))))
            localization = float(np.sort(energy)[-k:].sum() / max(np.sum(energy), 1e-12))

            order = np.argsort(z)
            z_order = z[order]
            r_order = residual[order]
            z_grid = np.linspace(-6, 6, 241)
            r_grid = np.interp(z_grid, z_order, r_order, left=np.nan, right=np.nan)
            valid = np.isfinite(r_grid)
            if valid.sum() > 5:
                rv = r_grid[valid]
                zv = z_grid[valid]
                first = np.gradient(rv, zv)
                second = np.gradient(first, zv)
                bend_energy = float(np.trapz(second**2, zv))
                mean_abs_bend = float(np.mean(np.abs(second)))
            else:
                bend_energy = 0.0
                mean_abs_bend = 0.0

            fill = np.nanmean(r_grid)
            r_fft = np.where(np.isfinite(r_grid), r_grid, fill)
            r_fft = r_fft - np.mean(r_fft)
            power = np.abs(np.fft.rfft(r_fft)) ** 2
            power[0] = 0
            total_power = float(power.sum())
            if total_power > 0:
                cutoff = max(2, int(0.20 * len(power)))
                low = float(power[1:cutoff].sum() / total_power)
                high = float(power[cutoff:].sum() / total_power)
                spectral_ratio = float(high / max(low, 1e-9))
            else:
                spectral_ratio = 0.0

            rows.append({
                "topology": topology,
                "label": TOPOLOGY_LABELS[topology],
                "n_modules": int(N),
                "mean_abs_residual": float(np.mean(abs_res)),
                "max_abs_residual": float(np.max(abs_res)),
                "total_residual_energy": float(np.sum(energy)),
                "residual_localization": localization,
                "residual_asymmetry": float((right_energy - left_energy) / lr_total) if lr_total > 0 else 0.0,
                "residual_entropy": normalized_entropy_from_energy(z, energy),
                "residual_bend_energy": bend_energy,
                "mean_abs_bend": mean_abs_bend,
                "residual_spectral_ratio": spectral_ratio,
            })

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_DIR / "residual_geometry_features.csv", index=False)
    return df

def build_embedding_from_features(feature_df):
    candidate_cols = [
        "mean_abs_residual",
        "max_abs_residual",
        "total_residual_energy",
        "residual_localization",
        "residual_asymmetry",
        "residual_entropy",
        "residual_bend_energy",
        "mean_abs_bend",
        "residual_spectral_ratio",
    ]
    feature_cols = [c for c in candidate_cols if c in feature_df.columns]
    X = feature_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)

    Xs = StandardScaler().fit_transform(X)
    pca = PCA(n_components=2)
    Z = pca.fit_transform(Xs)

    embedding = feature_df[["topology", "label", "n_modules"]].copy()
    embedding["pc1"] = Z[:, 0]
    embedding["pc2"] = Z[:, 1]
    embedding.to_csv(RESULTS_DIR / "residual_phase_trajectory_embedding.csv", index=False)

    return embedding, pca.explained_variance_ratio_, feature_cols

embedding_path = RESULTS_DIR / "residual_phase_trajectory_embedding.csv"
feature_path = RESULTS_DIR / "residual_geometry_features.csv"

if embedding_path.exists():
    traj_df = pd.read_csv(embedding_path)
    data_source = "loaded residual phase trajectory embedding"
    pca_variance = [np.nan, np.nan]
elif feature_path.exists():
    feature_df = pd.read_csv(feature_path)
    traj_df, pca_variance, feature_cols = build_embedding_from_features(feature_df)
    data_source = "rebuilt trajectory embedding from residual geometry features"
else:
    feature_df = regenerate_geometry_features()
    traj_df, pca_variance, feature_cols = build_embedding_from_features(feature_df)
    data_source = "regenerated residual geometry features and trajectory embedding internally"

# normalize expected names
if "n_modules" not in traj_df.columns:
    if "N" in traj_df.columns:
        traj_df["n_modules"] = traj_df["N"]
    elif "graph_size" in traj_df.columns:
        traj_df["n_modules"] = traj_df["graph_size"]

if "label" not in traj_df.columns:
    traj_df["label"] = traj_df["topology"].map(TOPOLOGY_LABELS).fillna(traj_df["topology"])
else:
    traj_df["label"] = traj_df["topology"].map(TOPOLOGY_LABELS).fillna(traj_df["label"])

traj_df = traj_df.sort_values(["topology", "n_modules"]).reset_index(drop=True)

print("data source:", data_source)
print("trajectory rows:", traj_df.shape)
traj_df.head()


## 2. Estimate topology-level residual fixed points

In [ ]:

def finite_size_model(N, x_inf, a, nu):
    N = np.asarray(N, dtype=float)
    return x_inf + a * np.power(N, -nu)

def fit_coordinate_fixed_point(N, values):
    N = np.asarray(N, dtype=float)
    values = np.asarray(values, dtype=float)

    try:
        p0 = [values[-1], values[0] - values[-1], 0.5]
        bounds = ([-50, -100, 0.05], [50, 100, 5.0])
        popt, pcov = curve_fit(
            finite_size_model,
            N,
            values,
            p0=p0,
            bounds=bounds,
            maxfev=20000,
        )
        pred = finite_size_model(N, *popt)
        rmse = float(np.sqrt(np.mean((pred - values) ** 2)))
        return {
            "x_inf": float(popt[0]),
            "a": float(popt[1]),
            "nu": float(popt[2]),
            "rmse": rmse,
            "method": "nonlinear_N_power",
        }
    except Exception:
        x = 1 / N
        coeff = np.polyfit(x[-3:], values[-3:], deg=1)
        x_inf = float(coeff[1])
        pred = np.polyval(coeff, x)
        rmse = float(np.sqrt(np.mean((pred - values) ** 2)))
        return {
            "x_inf": x_inf,
            "a": float(coeff[0]),
            "nu": 1.0,
            "rmse": rmse,
            "method": "linear_late_1_over_N",
        }

fixed_rows = []

for topology in TOPOLOGIES:
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    if len(sub) < 3:
        continue

    N = sub["n_modules"].to_numpy(dtype=float)
    x = sub["pc1"].to_numpy(dtype=float)
    y = sub["pc2"].to_numpy(dtype=float)

    fx = fit_coordinate_fixed_point(N, x)
    fy = fit_coordinate_fixed_point(N, y)

    x_inf = fx["x_inf"]
    y_inf = fy["x_inf"]
    final = np.array([x[-1], y[-1]])
    fixed = np.array([x_inf, y_inf])

    fixed_rows.append({
        "topology": topology,
        "label": TOPOLOGY_LABELS[topology],
        "x_inf": x_inf,
        "y_inf": y_inf,
        "nu_x": fx["nu"],
        "nu_y": fy["nu"],
        "a_x": fx["a"],
        "a_y": fy["a"],
        "rmse_x": fx["rmse"],
        "rmse_y": fy["rmse"],
        "fit_method_x": fx["method"],
        "fit_method_y": fy["method"],
        "distance_N128_to_fixed_point": float(np.linalg.norm(final - fixed)),
    })

fixed_df = pd.DataFrame(fixed_rows)
fixed_df.to_csv(RESULTS_DIR / "residual_fixed_point_estimates.csv", index=False)
fixed_df


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

N_grid = np.linspace(min(GRAPH_SIZES), 512, 300)

for topology in TOPOLOGIES:
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    fix = fixed_df[fixed_df["topology"] == topology]
    if sub.empty or fix.empty:
        continue

    row = fix.iloc[0]
    label = TOPOLOGY_LABELS[topology]

    axes[0].plot(sub["n_modules"], sub["pc1"], marker="o", label=label)
    axes[1].plot(sub["n_modules"], sub["pc2"], marker="o", label=label)

    x_pred = finite_size_model(N_grid, row["x_inf"], row["a_x"], row["nu_x"])
    y_pred = finite_size_model(N_grid, row["y_inf"], row["a_y"], row["nu_y"])
    axes[0].plot(N_grid, x_pred, linestyle="--", alpha=0.8)
    axes[1].plot(N_grid, y_pred, linestyle="--", alpha=0.8)

    axes[0].scatter([512], [row["x_inf"]], marker="*", s=120, color="black", alpha=0.8)
    axes[1].scatter([512], [row["y_inf"]], marker="*", s=120, color="black", alpha=0.8)

axes[0].set_title("PC1 finite-size extrapolation")
axes[1].set_title("PC2 finite-size extrapolation")

for ax in axes:
    ax.set_xlabel("graph size N")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("residual coordinate")
axes[1].legend(fontsize=8, loc="best")

plt.suptitle("Residual fixed-point coordinate extrapolation")
plt.tight_layout()

fig_path = FIG_DIR / "residual_fixed_point_extrapolation.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## 3. Distance-to-fixed-point scaling

In [ ]:

distance_rows = []

for topology in TOPOLOGIES:
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    fix = fixed_df[fixed_df["topology"] == topology]
    if sub.empty or fix.empty:
        continue

    row = fix.iloc[0]
    fixed = np.array([row["x_inf"], row["y_inf"]], dtype=float)

    for _, pt in sub.iterrows():
        p = np.array([pt["pc1"], pt["pc2"]], dtype=float)
        distance_rows.append({
            "topology": topology,
            "label": TOPOLOGY_LABELS[topology],
            "n_modules": int(pt["n_modules"]),
            "distance_to_fixed_point": float(np.linalg.norm(p - fixed)),
        })

distance_df = pd.DataFrame(distance_rows)

def power_decay(N, C, nu):
    return C * np.power(np.asarray(N, dtype=float), -nu)

decay_rows = []

for topology in TOPOLOGIES:
    sub = distance_df[distance_df["topology"] == topology].sort_values("n_modules")
    if len(sub) < 3:
        continue

    N = sub["n_modules"].to_numpy(dtype=float)
    d = np.maximum(sub["distance_to_fixed_point"].to_numpy(dtype=float), 1e-9)

    try:
        popt, _ = curve_fit(
            power_decay,
            N,
            d,
            p0=[d[0] * N[0] ** 0.5, 0.5],
            bounds=([1e-9, 0.01], [1000, 10]),
            maxfev=20000,
        )
        pred = power_decay(N, *popt)
        rmse = float(np.sqrt(np.mean((pred - d) ** 2)))
        decay_rows.append({
            "topology": topology,
            "label": TOPOLOGY_LABELS[topology],
            "C": float(popt[0]),
            "nu_distance": float(popt[1]),
            "distance_fit_rmse": rmse,
        })
    except Exception:
        decay_rows.append({
            "topology": topology,
            "label": TOPOLOGY_LABELS[topology],
            "C": np.nan,
            "nu_distance": np.nan,
            "distance_fit_rmse": np.nan,
        })

decay_df = pd.DataFrame(decay_rows)
distance_df.to_csv(RESULTS_DIR / "residual_distance_to_fixed_point.csv", index=False)
decay_df.to_csv(RESULTS_DIR / "residual_fixed_point_distance_scaling.csv", index=False)

plt.figure(figsize=(9, 6))

N_grid = np.linspace(min(GRAPH_SIZES), max(GRAPH_SIZES), 300)

for topology in TOPOLOGIES:
    sub = distance_df[distance_df["topology"] == topology].sort_values("n_modules")
    fit = decay_df[decay_df["topology"] == topology]
    if sub.empty:
        continue

    plt.plot(
        sub["n_modules"],
        sub["distance_to_fixed_point"],
        marker="o",
        linewidth=2,
        label=TOPOLOGY_LABELS[topology],
    )

    if not fit.empty and np.isfinite(fit.iloc[0]["C"]):
        C = fit.iloc[0]["C"]
        nu = fit.iloc[0]["nu_distance"]
        plt.plot(N_grid, power_decay(N_grid, C, nu), linestyle="--", alpha=0.6)

plt.xlabel("graph size N")
plt.ylabel("distance to fixed-point estimate")
plt.title("Distance-to-fixed-point finite-size scaling")
plt.grid(alpha=0.3)
plt.legend(fontsize=9)

fig_path = FIG_DIR / "distance_to_fixed_point_scaling.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

decay_df


## 4. Residual beta-flow vectors

In [ ]:

beta_rows = []

for topology in TOPOLOGIES:
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    points = sub[["pc1", "pc2"]].to_numpy(dtype=float)
    sizes = sub["n_modules"].to_numpy(dtype=int)

    for i in range(len(points) - 1):
        beta = points[i + 1] - points[i]
        beta_rows.append({
            "topology": topology,
            "label": TOPOLOGY_LABELS[topology],
            "from_N": int(sizes[i]),
            "to_N": int(sizes[i + 1]),
            "transition": f"{int(sizes[i])}→{int(sizes[i + 1])}",
            "pc1": float(points[i, 0]),
            "pc2": float(points[i, 1]),
            "beta_x": float(beta[0]),
            "beta_y": float(beta[1]),
            "beta_magnitude": float(np.linalg.norm(beta)),
        })

beta_df = pd.DataFrame(beta_rows)
beta_df.to_csv(RESULTS_DIR / "residual_beta_flow.csv", index=False)

plt.figure(figsize=(10, 8))

for topology in TOPOLOGIES:
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    beta_sub = beta_df[beta_df["topology"] == topology].sort_values("from_N")

    if sub.empty:
        continue

    plt.plot(sub["pc1"], sub["pc2"], marker="o", linewidth=1.8, label=TOPOLOGY_LABELS[topology])

    for _, row in beta_sub.iterrows():
        plt.arrow(
            row["pc1"],
            row["pc2"],
            row["beta_x"] * 0.85,
            row["beta_y"] * 0.85,
            head_width=0.08,
            length_includes_head=True,
            alpha=0.7,
        )

    for _, row in sub.iterrows():
        plt.annotate(
            f"N={int(row['n_modules'])}",
            xy=(row["pc1"], row["pc2"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )

plt.axhline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.axvline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.xlabel("residual manifold coordinate 1")
plt.ylabel("residual manifold coordinate 2")
plt.title("Residual beta-flow vectors")
plt.grid(alpha=0.3)
plt.legend(fontsize=9, loc="best")

fig_path = FIG_DIR / "residual_beta_flow_vectors.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

beta_df.head()


## 5. Fixed-point basin map

In [ ]:

plt.figure(figsize=(10, 8))

for topology in TOPOLOGIES:
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    fix = fixed_df[fixed_df["topology"] == topology]
    if sub.empty or fix.empty:
        continue

    row = fix.iloc[0]
    label = TOPOLOGY_LABELS[topology]

    plt.plot(sub["pc1"], sub["pc2"], marker="o", linewidth=2, label=label)
    plt.scatter(row["x_inf"], row["y_inf"], marker="*", s=220, color="black", zorder=5)

    final = sub.iloc[-1]
    plt.plot(
        [final["pc1"], row["x_inf"]],
        [final["pc2"], row["y_inf"]],
        linestyle=":",
        linewidth=1.4,
        color="black",
        alpha=0.8,
    )

    plt.annotate(
        f"{label} fixed-point estimate",
        xy=(row["x_inf"], row["y_inf"]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=8,
        alpha=0.85,
    )

plt.axhline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.axvline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.xlabel("residual manifold coordinate 1")
plt.ylabel("residual manifold coordinate 2")
plt.title("Residual fixed-point basin map")
plt.grid(alpha=0.3)
plt.legend(fontsize=9, loc="best")

fig_path = FIG_DIR / "residual_fixed_point_basin_map.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## 6. Universality branch fixed points

In [ ]:

cluster_path = RESULTS_DIR / "universality_clusters.csv"

if cluster_path.exists():
    cluster_df = pd.read_csv(cluster_path)
else:
    descriptor_rows = []
    for topology in TOPOLOGIES:
        sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
        pts = sub[["pc1", "pc2"]].to_numpy(dtype=float)
        if len(pts) < 2:
            continue
        diffs = np.diff(pts, axis=0)
        length = float(np.sum(np.linalg.norm(diffs, axis=1)))
        endpoint = float(np.linalg.norm(pts[-1] - pts[0]))
        curvature = 0.0
        for i in range(1, len(pts) - 1):
            a = pts[i] - pts[i - 1]
            b = pts[i + 1] - pts[i]
            denom = np.linalg.norm(a) * np.linalg.norm(b)
            if denom > 0:
                curvature += math.acos(float(np.clip(np.dot(a, b) / denom, -1, 1)))
        descriptor_rows.append({
            "topology": topology,
            "label": TOPOLOGY_LABELS[topology],
            "trajectory_length": length,
            "endpoint_distance": endpoint,
            "turning_angle_sum": curvature,
            "tortuosity": length / max(endpoint, 1e-9),
        })

    desc = pd.DataFrame(descriptor_rows)
    X = StandardScaler().fit_transform(desc[["trajectory_length", "endpoint_distance", "turning_angle_sum", "tortuosity"]])
    D = pairwise_distances(X)
    D = 0.5 * (D + D.T)
    np.fill_diagonal(D, 0.0)
    Z = linkage(squareform(D, checks=False), method="average")
    clusters = fcluster(Z, t=2, criterion="maxclust")

    cluster_df = desc[["topology", "label"]].copy()
    cluster_df["universality_cluster"] = clusters.astype(int)
    cluster_df.to_csv(cluster_path, index=False)

cluster_df["label"] = cluster_df["topology"].map(TOPOLOGY_LABELS).fillna(cluster_df.get("label", cluster_df["topology"]))

branch_rows = []

for cluster_id, members in cluster_df.groupby("universality_cluster"):
    member_topologies = list(members["topology"])
    pooled = traj_df[traj_df["topology"].isin(member_topologies)].copy()

    branch_curve = (
        pooled.groupby("n_modules")[["pc1", "pc2"]]
        .mean()
        .reset_index()
        .sort_values("n_modules")
    )

    if len(branch_curve) < 3:
        continue

    N = branch_curve["n_modules"].to_numpy(dtype=float)
    x = branch_curve["pc1"].to_numpy(dtype=float)
    y = branch_curve["pc2"].to_numpy(dtype=float)

    fx = fit_coordinate_fixed_point(N, x)
    fy = fit_coordinate_fixed_point(N, y)

    branch_rows.append({
        "universality_cluster": int(cluster_id),
        "member_topologies": ",".join(member_topologies),
        "member_labels": ", ".join([TOPOLOGY_LABELS[t] for t in member_topologies]),
        "x_inf": fx["x_inf"],
        "y_inf": fy["x_inf"],
        "nu_x": fx["nu"],
        "nu_y": fy["nu"],
        "rmse_x": fx["rmse"],
        "rmse_y": fy["rmse"],
    })

branch_df = pd.DataFrame(branch_rows)
branch_df.to_csv(RESULTS_DIR / "universality_branch_fixed_points.csv", index=False)

plt.figure(figsize=(10, 7))

for cluster_id, members in cluster_df.groupby("universality_cluster"):
    member_topologies = list(members["topology"])
    pooled = traj_df[traj_df["topology"].isin(member_topologies)].copy()

    for topology in member_topologies:
        sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
        plt.plot(sub["pc1"], sub["pc2"], marker="o", alpha=0.35, linewidth=1.3)

    branch_curve = (
        pooled.groupby("n_modules")[["pc1", "pc2"]]
        .mean()
        .reset_index()
        .sort_values("n_modules")
    )

    plt.plot(
        branch_curve["pc1"],
        branch_curve["pc2"],
        marker="o",
        linewidth=3,
        label=f"cluster {cluster_id}: " + ", ".join([TOPOLOGY_LABELS[t] for t in member_topologies]),
    )

    fix = branch_df[branch_df["universality_cluster"] == cluster_id]
    if not fix.empty:
        row = fix.iloc[0]
        plt.scatter(row["x_inf"], row["y_inf"], marker="*", s=260, color="black", zorder=6)
        plt.annotate(
            f"cluster {cluster_id} fixed-point estimate",
            xy=(row["x_inf"], row["y_inf"]),
            xytext=(8, 8),
            textcoords="offset points",
            fontsize=9,
            alpha=0.85,
        )

plt.axhline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.axvline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.xlabel("residual manifold coordinate 1")
plt.ylabel("residual manifold coordinate 2")
plt.title("Universality branch fixed-point estimates")
plt.grid(alpha=0.3)
plt.legend(fontsize=8, loc="best")

fig_path = FIG_DIR / "universality_branch_fixed_points.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

branch_df


## 7. Finite-size collapse quality and locking status

In [ ]:

quality_rows = []

for topology in TOPOLOGIES:
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    fix = fixed_df[fixed_df["topology"] == topology]
    decay = decay_df[decay_df["topology"] == topology]
    beta_late = beta_df[
        (beta_df["topology"] == topology)
        & (beta_df["from_N"] == 64)
        & (beta_df["to_N"] == 128)
    ]

    if sub.empty or fix.empty:
        continue

    row = fix.iloc[0]
    d_final = float(row["distance_N128_to_fixed_point"])
    late_beta = float(beta_late.iloc[0]["beta_magnitude"]) if not beta_late.empty else np.nan

    rmse = float(np.sqrt(row["rmse_x"]**2 + row["rmse_y"]**2))
    total_span = float(
        np.linalg.norm(
            sub[["pc1", "pc2"]].to_numpy(dtype=float).max(axis=0)
            - sub[["pc1", "pc2"]].to_numpy(dtype=float).min(axis=0)
        )
    )
    collapse_quality = float(1 - rmse / max(total_span, 1e-9))
    collapse_quality = float(np.clip(collapse_quality, 0, 1))

    if collapse_quality >= 0.85 and late_beta < 0.75:
        status = "locked"
    elif collapse_quality >= 0.70 and late_beta < 1.25:
        status = "approaching"
    elif late_beta >= 1.75:
        status = "drifting"
    else:
        status = "ambiguous"

    quality_rows.append({
        "topology": topology,
        "label": TOPOLOGY_LABELS[topology],
        "collapse_quality": collapse_quality,
        "late_stage_locking": late_beta,
        "distance_N128_to_fixed_point": d_final,
        "fixed_point_status": status,
    })

quality_df = pd.DataFrame(quality_rows)
quality_df.to_csv(RESULTS_DIR / "fixed_point_collapse_quality.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

plot_df = quality_df.sort_values("collapse_quality")
axes[0].barh(plot_df["label"], plot_df["collapse_quality"])
axes[0].set_title("fixed-point collapse quality")
axes[0].set_xlim(0, 1)
axes[0].grid(alpha=0.3, axis="x")

plot_df = quality_df.sort_values("late_stage_locking")
axes[1].barh(plot_df["label"], plot_df["late_stage_locking"])
axes[1].set_title("late-stage beta flow |64→128|")
axes[1].grid(alpha=0.3, axis="x")

plt.suptitle("Fixed-point collapse quality and late-stage locking")
plt.tight_layout()

fig_path = FIG_DIR / "fixed_point_collapse_quality.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

quality_df


## 8. Fixed-point summary table

In [ ]:

summary_df = (
    fixed_df
    .merge(decay_df[["topology", "nu_distance", "distance_fit_rmse"]], on="topology", how="left")
    .merge(quality_df[["topology", "collapse_quality", "late_stage_locking", "fixed_point_status"]], on="topology", how="left")
    .merge(cluster_df[["topology", "universality_cluster"]], on="topology", how="left")
)

summary_df = summary_df[
    [
        "topology",
        "label",
        "universality_cluster",
        "x_inf",
        "y_inf",
        "nu_x",
        "nu_y",
        "nu_distance",
        "distance_N128_to_fixed_point",
        "late_stage_locking",
        "collapse_quality",
        "fixed_point_status",
        "rmse_x",
        "rmse_y",
        "distance_fit_rmse",
    ]
].sort_values(["universality_cluster", "label"]).reset_index(drop=True)

summary_df.to_csv(RESULTS_DIR / "residual_fixed_point_summary.csv", index=False)
summary_df


## 9. Interpretation export

In [ ]:

status_counts = summary_df["fixed_point_status"].value_counts().to_dict()

best_quality = summary_df.sort_values("collapse_quality", ascending=False).iloc[0]
lowest_late_beta = summary_df.sort_values("late_stage_locking", ascending=True).iloc[0]
largest_late_beta = summary_df.sort_values("late_stage_locking", ascending=False).iloc[0]

interpretation = {
    "notebook": "21_residual_fixed_points_finite_size_collapse.ipynb",
    "core_question": "Do residual universality trajectories converge toward fixed points as graph size increases?",
    "core_claim": (
        "Residual topology classes can be studied as finite-size flows toward "
        "topology-specific or branch-specific fixed-point estimates."
    ),
    "data_source": data_source,
    "status_counts": status_counts,
    "highest_collapse_quality": {
        "topology": str(best_quality["topology"]),
        "label": str(best_quality["label"]),
        "collapse_quality": float(best_quality["collapse_quality"]),
    },
    "smallest_late_stage_beta_flow": {
        "topology": str(lowest_late_beta["topology"]),
        "label": str(lowest_late_beta["label"]),
        "late_stage_locking": float(lowest_late_beta["late_stage_locking"]),
    },
    "largest_late_stage_beta_flow": {
        "topology": str(largest_late_beta["topology"]),
        "label": str(largest_late_beta["label"]),
        "late_stage_locking": float(largest_late_beta["late_stage_locking"]),
    },
    "figures": [
        "residual_fixed_point_extrapolation.png",
        "distance_to_fixed_point_scaling.png",
        "residual_beta_flow_vectors.png",
        "residual_fixed_point_basin_map.png",
        "universality_branch_fixed_points.png",
        "fixed_point_collapse_quality.png",
    ],
    "results": [
        "residual_fixed_point_estimates.csv",
        "residual_distance_to_fixed_point.csv",
        "residual_fixed_point_distance_scaling.csv",
        "residual_beta_flow.csv",
        "universality_branch_fixed_points.csv",
        "fixed_point_collapse_quality.csv",
        "residual_fixed_point_summary.csv",
        "residual_fixed_point_interpretation.json",
    ],
}

json_path = RESULTS_DIR / "residual_fixed_point_interpretation.json"
json_path.write_text(json.dumps(interpretation, indent=2), encoding="utf-8")

md = [
    "# Notebook 21 — Residual Fixed Points and Finite-Size Universality Collapse",
    "",
    "## Core question",
    "",
    "Do residual universality trajectories converge toward fixed points as graph size increases?",
    "",
    "## Core claim",
    "",
    "Residual topology classes can be studied as finite-size flows toward topology-specific or branch-specific fixed-point estimates.",
    "",
    "## Recommended figures",
    "",
    "- `figures/residual_fixed_point_basin_map.png`",
    "- `figures/distance_to_fixed_point_scaling.png`",
    "- `figures/residual_beta_flow_vectors.png`",
    "- `figures/universality_branch_fixed_points.png`",
    "- `figures/fixed_point_collapse_quality.png`",
    "",
    "## Key computed observations",
    "",
    f"- Highest fixed-point collapse quality: `{best_quality['label']}` "
    f"(quality ≈ {best_quality['collapse_quality']:.3f}).",
    f"- Smallest late-stage beta flow: `{lowest_late_beta['label']}` "
    f"(|β_64→128| ≈ {lowest_late_beta['late_stage_locking']:.3f}).",
    f"- Largest late-stage beta flow: `{largest_late_beta['label']}` "
    f"(|β_64→128| ≈ {largest_late_beta['late_stage_locking']:.3f}).",
    "",
    "## Status labels",
    "",
]

for status, count in status_counts.items():
    md.append(f"- `{status}`: {count}")

md.append("")
md.append("## Note")
md.append("")
md.append("These are finite-size fixed-point estimates, not proof-level fixed points.")

md_path = DOCS_DIR / "notebook_21_residual_fixed_points.md"
md_path.write_text("\n".join(md), encoding="utf-8")

print(json.dumps(interpretation, indent=2))
print("saved:", json_path)
print("saved:", md_path)


## Optional export zip / download

In [ ]:

zip_path = REPO_ROOT / "notebook_21_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        if folder.exists():
            for file in folder.glob("*"):
                if file.is_file():
                    zf.write(file)

print(f"Created: {zip_path}")

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))
